# Diagnostic U-Net 256 Threshold Sweep

This notebook is a short diagnostic run for checking why the initial U-Net baseline can produce zero Dice at threshold 0.5.

It is **not** used as the final augmentation comparison result. It keeps the baseline setup lightweight and adds threshold/probability diagnostics only.


In [ ]:
# Runtime setup
from google.colab import drive

drive.mount('/content/drive')

REPO_URL = 'https://github.com/dansojo/Industrial-Defect-Segmentation.git'
REPO_DIR = '/content/Industrial-Defect-Segmentation'
DRIVE_DATA_ARCHIVES = [
    '/content/drive/MyDrive/VisA_segmentation/VisA.tar',
    '/content/drive/MyDrive/VisA_segmentation/VisA.zip',
    '/content/drive/MyDrive/VisA_segmentation/VisA.tar.gz',
]
DATA_PARENT = '/content/data'
RESULTS_ROOT = '/content/drive/MyDrive/VisA_segmentation/visa_results/colab_runs'


In [ ]:
# Clone or update repository
import os
import subprocess
from pathlib import Path

if not Path(REPO_DIR).exists():
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)

os.chdir(REPO_DIR)
print('Repo:', Path.cwd())


In [ ]:
# Install dependencies
import subprocess
import sys

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'albumentations', 'opencv-python-headless', 'pandas', 'matplotlib', 'tqdm'
], check=True)


In [ ]:
# Prepare dataset archive from Drive into Colab local disk
import tarfile
import zipfile
from pathlib import Path

DATA_PARENT_PATH = Path(DATA_PARENT)
DATA_PARENT_PATH.mkdir(parents=True, exist_ok=True)
archive_candidates = [Path(p) for p in DRIVE_DATA_ARCHIVES]


def find_data_root():
    candidates = [
        DATA_PARENT_PATH / 'VisA',
        DATA_PARENT_PATH / 'VisA_data',
        DATA_PARENT_PATH,
    ]
    return next(
        (p for p in candidates if (p / 'split_csv').exists() and (p / 'candle').exists()),
        None,
    )


def extract_archive(archive_path: Path):
    suffixes = ''.join(archive_path.suffixes).lower()
    print(f'Extracting {archive_path} to {DATA_PARENT_PATH} ...')
    if suffixes.endswith('.zip'):
        with zipfile.ZipFile(archive_path, 'r') as zf:
            zf.extractall(DATA_PARENT_PATH)
    elif suffixes.endswith('.tar') or suffixes.endswith('.tar.gz') or suffixes.endswith('.tgz'):
        with tarfile.open(archive_path, 'r:*') as tf:
            tf.extractall(DATA_PARENT_PATH)
    else:
        raise ValueError(f'Unsupported archive format: {archive_path}')


DATA_ROOT = find_data_root()
if DATA_ROOT is None:
    archive_path = next((p for p in archive_candidates if p.exists()), None)
    if archive_path is None:
        checked = '\n'.join(str(p) for p in archive_candidates)
        raise FileNotFoundError(f'No VisA archive found. Checked:\n{checked}')
    extract_archive(archive_path)
    DATA_ROOT = find_data_root()

if DATA_ROOT is None:
    children = [str(p) for p in DATA_PARENT_PATH.iterdir()]
    raise FileNotFoundError(f'Could not find expected VisA dataset root. children={children}')

print('DATA_ROOT:', DATA_ROOT)
print('Top-level sample:', sorted([p.name for p in DATA_ROOT.iterdir()])[:8])


In [ ]:
# Experiment identity
EXPERIMENT_NAME = 'diagnostic_unet_256_aug_none_threshold_sweep'
AUGMENTATION_NAME = 'aug_none'


In [ ]:
# Imports
import json
import random
import time
from collections import defaultdict
from pathlib import Path

import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from src.datasets.visa_dataset import VisASegmentationDataset
from src.losses.losses import build_loss
from src.metrics import binary_stats, merge_stats, metrics_from_stats
from src.models.unet import build_unet
from src.visualize import IMAGENET_MEAN, IMAGENET_STD, overlay_mask, tensor_to_rgb

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# Diagnostic config
INPUT_SIZE = 256
BATCH_SIZE = 16
EPOCHS = 5
LR = 1e-3
LOSS_NAME = 'bce_dice'
THRESHOLD = 0.5
DIAGNOSTIC_THRESHOLDS = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5]
NUM_WORKERS = 0

SPLIT_CSV = Path(REPO_DIR) / 'data' / 'splits' / 'visa_2cls_highshot_train_val_test.csv'
RUN_DIR = Path(RESULTS_ROOT) / EXPERIMENT_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

config = {
    'experiment_name': EXPERIMENT_NAME,
    'augmentation': AUGMENTATION_NAME,
    'input_size': INPUT_SIZE,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'lr': LR,
    'loss': LOSS_NAME,
    'threshold': THRESHOLD,
    'diagnostic_thresholds': DIAGNOSTIC_THRESHOLDS,
    'num_workers': NUM_WORKERS,
    'split_csv': str(SPLIT_CSV),
    'data_root': str(DATA_ROOT),
    'purpose': 'diagnose zero Dice / background collapse without changing baseline training setup',
}
(RUN_DIR / 'config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
print(json.dumps(config, indent=2))


In [ ]:
# Transforms: diagnostic run uses resize-only baseline
IMAGENET_MEAN_TUPLE = (0.485, 0.456, 0.406)
IMAGENET_STD_TUPLE = (0.229, 0.224, 0.225)


def get_transforms(split: str):
    return A.Compose([
        A.Resize(INPUT_SIZE, INPUT_SIZE),
        A.Normalize(mean=IMAGENET_MEAN_TUPLE, std=IMAGENET_STD_TUPLE),
        ToTensorV2(),
    ])


In [ ]:
# Datasets and dataloaders
train_ds = VisASegmentationDataset(DATA_ROOT, SPLIT_CSV, split='train', transform=get_transforms('train'))
val_ds = VisASegmentationDataset(DATA_ROOT, SPLIT_CSV, split='val', transform=get_transforms('val'))
test_ds = VisASegmentationDataset(DATA_ROOT, SPLIT_CSV, split='test', transform=get_transforms('test'))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print('Train:', len(train_ds), 'Val:', len(val_ds), 'Test:', len(test_ds))
print(pd.read_csv(SPLIT_CSV).groupby(['split','label']).size().unstack(fill_value=0))


def describe_loader_masks(loader, name, max_batches=8):
    mask_pixels = 0.0
    total_pixels = 0.0
    labels = []
    for batch_idx, batch in enumerate(loader):
        masks = batch['mask']
        mask_pixels += float((masks > 0.5).sum().item())
        total_pixels += float(masks.numel())
        labels.extend(list(batch['label']))
        if batch_idx + 1 >= max_batches:
            break
    label_counts = pd.Series(labels).value_counts().to_dict()
    positive_ratio = mask_pixels / max(total_pixels, 1.0)
    print(f'{name} mask_positive_ratio(first {max_batches} batches): {positive_ratio:.8f} | labels: {label_counts}')


describe_loader_masks(train_loader, 'train')
describe_loader_masks(val_loader, 'val')


In [ ]:
# Model, loss, optimizer
model = build_unet().to(device)
criterion = build_loss(LOSS_NAME)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)


def train_one_epoch(loader):
    model.train()
    losses = []
    for batch in tqdm(loader, leave=False):
        images = batch['image'].to(device, non_blocking=True)
        masks = batch['mask'].to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return float(np.mean(losses))


@torch.no_grad()
def evaluate_with_sweep(loader):
    model.eval()
    losses = []
    threshold_stats = {thr: [] for thr in DIAGNOSTIC_THRESHOLDS}
    prob_means = []
    prob_maxes = []
    prob_percentiles = []
    target_positive_pixels = 0.0
    total_pixels = 0.0

    for batch in tqdm(loader, leave=False):
        images = batch['image'].to(device, non_blocking=True)
        masks = batch['mask'].to(device, non_blocking=True)
        logits = model(images)
        loss = criterion(logits, masks)
        losses.append(loss.item())

        probs = torch.sigmoid(logits.detach())
        prob_means.append(float(probs.mean().item()))
        prob_maxes.append(float(probs.max().item()))
        flat = probs.detach().flatten()
        prob_percentiles.append(torch.quantile(flat, torch.tensor([0.5, 0.9, 0.99], device=flat.device)).cpu().numpy())

        target_positive_pixels += float((masks >= 0.5).sum().item())
        total_pixels += float(masks.numel())

        for thr in DIAGNOSTIC_THRESHOLDS:
            threshold_stats[thr].append(binary_stats(logits.cpu(), masks.cpu(), threshold=thr))

    rows = []
    for thr, items in threshold_stats.items():
        merged = merge_stats(items)
        metrics = metrics_from_stats(merged)
        rows.append({'threshold': thr, **metrics})

    sweep_df = pd.DataFrame(rows)
    percentile_array = np.stack(prob_percentiles, axis=0)
    diagnostics = {
        'loss': float(np.mean(losses)),
        'pred_prob_mean': float(np.mean(prob_means)),
        'pred_prob_max': float(np.max(prob_maxes)),
        'pred_prob_p50': float(percentile_array[:, 0].mean()),
        'pred_prob_p90': float(percentile_array[:, 1].mean()),
        'pred_prob_p99': float(percentile_array[:, 2].mean()),
        'target_positive_ratio': target_positive_pixels / max(total_pixels, 1.0),
    }
    return diagnostics, sweep_df


In [ ]:
# Train short diagnostic run
history = []
best_dice_at_05 = -1.0
best_path = RUN_DIR / 'best_model.pt'

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(train_loader)
    val_diag, val_sweep = evaluate_with_sweep(val_loader)
    dice_at_05 = float(val_sweep.loc[val_sweep['threshold'] == 0.5, 'dice'].iloc[0])

    row = {'epoch': epoch, 'train_loss': train_loss, **{f'val_{k}': v for k, v in val_diag.items()}}
    for sweep_row in val_sweep.to_dict('records'):
        thr = str(sweep_row['threshold']).replace('.', 'p')
        row[f'val_dice_thr_{thr}'] = sweep_row['dice']
        row[f'val_recall_thr_{thr}'] = sweep_row['recall']
    history.append(row)
    pd.DataFrame(history).to_csv(RUN_DIR / 'diagnostic_metrics.csv', index=False)
    val_sweep.to_csv(RUN_DIR / f'threshold_sweep_val_epoch_{epoch:02d}.csv', index=False)

    if dice_at_05 > best_dice_at_05:
        best_dice_at_05 = dice_at_05
        torch.save({'model_state_dict': model.state_dict(), 'config': config, 'epoch': epoch}, best_path)

    print(f"Epoch {epoch:02d}/{EPOCHS} | train_loss={train_loss:.4f} | val_loss={val_diag['loss']:.4f} | prob_mean={val_diag['pred_prob_mean']:.4f} | prob_max={val_diag['pred_prob_max']:.4f} | p99={val_diag['pred_prob_p99']:.4f} | target_pos={val_diag['target_positive_ratio']:.8f}")
    display(val_sweep)

print('Saved diagnostic run to:', RUN_DIR)


In [ ]:
# Final validation/test threshold sweep
checkpoint = torch.load(best_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])

val_diag, val_sweep = evaluate_with_sweep(val_loader)
test_diag, test_sweep = evaluate_with_sweep(test_loader)

pd.DataFrame([{'split': 'val', **val_diag}, {'split': 'test', **test_diag}]).to_csv(RUN_DIR / 'diagnostic_probability_stats.csv', index=False)
val_sweep.to_csv(RUN_DIR / 'threshold_sweep_val.csv', index=False)
test_sweep.to_csv(RUN_DIR / 'threshold_sweep_test.csv', index=False)

print('Probability diagnostics')
display(pd.DataFrame([{'split': 'val', **val_diag}, {'split': 'test', **test_diag}]))
print('Validation threshold sweep')
display(val_sweep)
print('Test threshold sweep')
display(test_sweep)


In [ ]:
# Save prediction heatmap grid for visual diagnosis
@torch.no_grad()
def save_heatmap_grid(loader, output_path, max_items=6):
    model.eval()
    batch = next(iter(loader))
    images = batch['image'].to(device)
    masks = batch['mask'].to(device)
    logits = model(images)
    probs = torch.sigmoid(logits).cpu()
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    tiles = []
    count = min(max_items, images.shape[0])
    for idx in range(count):
        image_rgb = tensor_to_rgb(batch['image'][idx].cpu())
        gt_mask = batch['mask'][idx, 0].cpu().numpy() > 0.5
        pred_05 = probs[idx, 0].numpy() >= 0.5
        heat = (probs[idx, 0].numpy() * 255).astype(np.uint8)
        heat_color = cv2.applyColorMap(heat, cv2.COLORMAP_JET)
        heat_color = cv2.cvtColor(heat_color, cv2.COLOR_BGR2RGB)
        gt_overlay = overlay_mask(image_rgb, gt_mask, color=(0, 255, 0), alpha=0.45)
        pred_overlay = overlay_mask(image_rgb, pred_05, color=(255, 0, 0), alpha=0.45)
        tile = np.concatenate([image_rgb, gt_overlay, pred_overlay, heat_color], axis=1)
        tiles.append(tile)

    grid = np.concatenate(tiles, axis=0)
    cv2.imwrite(str(output_path), cv2.cvtColor(grid, cv2.COLOR_RGB2BGR))
    return output_path

val_heatmap_path = save_heatmap_grid(val_loader, RUN_DIR / 'val_heatmap_grid.jpg')
test_heatmap_path = save_heatmap_grid(test_loader, RUN_DIR / 'test_heatmap_grid.jpg')
print('Saved:', val_heatmap_path)
print('Saved:', test_heatmap_path)


In [ ]:
# Display heatmap grids
from IPython.display import Image, display

print('Columns: original | GT green | pred@0.5 red | probability heatmap')
display(Image(filename=str(RUN_DIR / 'val_heatmap_grid.jpg')))
display(Image(filename=str(RUN_DIR / 'test_heatmap_grid.jpg')))


## How to interpret this diagnostic notebook

Use this notebook only to answer these questions:

1. Does the validation set contain positive mask pixels after transforms? Check `target_positive_ratio`.
2. Does the model output probabilities below 0.5 but above lower thresholds? Check `pred_prob_max`, `pred_prob_p99`, and threshold sweep rows.
3. Does Dice revive at thresholds such as 0.05, 0.1, or 0.2?
4. Is there spatial signal near defects in the probability heatmap?

Do **not** use this notebook to choose the final threshold. Threshold selection belongs to a later tuning stage after the augmentation baseline is decided.
